# <font color=#ef8b35> **Data Visualization: gráficos de comparação e distribuição**

---

#### <font color=#f7c59a> Desafio referente à _Aula 1._ "Comparando dados" do curso **"Data Visualization: gráficos de comparação e distribuição"**, da *Alura*.

Vamos praticar a criação de gráficos de comparação (colunas e barras) que aprendemos até aqui. Para a prática, vamos seguir utilizando o conjunto de dados do relatório de vendas das lojas de departamentos de 2016 a 2019 que está disponível no [github do projeto](https://github.com/alura-cursos/dataviz-graficos/blob/master/dados/relatorio_vendas.csv).

Neste desafio, a missão é construir as visualizações que respondam aos questionamentos que compartilharemos aqui abaixo:

<font color=#ef8b35> **Desafio 1:**</font> Quais são os lucros das vendas por ano? Em qual ano obtivemos o maior lucro?

<font color=#ef8b35> **Desafio 2:**</font> Qual foi o faturamento (vendas) dos top 10 produtos durante o período de 2016 a 2019 do nosso conjunto de dados? Adicione um pequeno texto falando dos 3 produtos que mais venderam.

### <font color=#f2a25d> **Importando os dados**

In [ ]:
# Importa as biblitecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Importa os dados
vendas = pd.read_csv('relatorio_vendas.csv')
# Transforma as colunas de data em Datetime
vendas['data_pedido'] = pd.to_datetime(vendas['data_pedido'], format='%Y-%m-%d')
vendas['data_envio'] = pd.to_datetime(vendas['data_envio'], format='%Y-%m-%d')

## <font color=a76125> **Desafio 1**

##### <font color=#f8d0ae> **Criando o _Dataframe_**

In [ ]:
# Cria o dataframe para lucro anual
lucro_ano = vendas.copy()
# Seleciona somente as colunas de interesse
lucro_ano = lucro_ano[['data_pedido','lucro']]

# Cria a coluna com os anos
lucro_ano['ano'] = lucro_ano.data_pedido.dt.year
# Remove a coluna 'data_pedido'
lucro_ano.drop(labels='data_pedido', axis=1, inplace=True)

# Agrupa o lucro por ano
lucro_ano = lucro_ano.groupby(['ano']).agg('sum')


##### <font color=#f8d0ae> **Listando as cores da paleta**

In [ ]:
# Paleta de cores
VERDE1, VERDE2, VERDE3 = '#9bd6c5', '#59bb9f', '#35705f'
CINZA1, CINZA2, CINZA3, CINZA4, CINZA5 = '#212529', '#495057', '#adb5bd', '#dee2e6', '#f8f9fa'
LARANJA1, AMARELO1, VERMELHO1 = '#e99d56','#fae690', '#b04324'
AZUL1, AZUL2, AZUL3 = '#a1b3c4', '#7a93ab', '#556677'


##### <font color=#f8d0ae> **Gerando o gráfico com uma função**

In [ ]:
def criar_grafico_colunas(cores:list=[VERDE2]):

    # Área do gráfico e tema de visualização
    fig, ax = plt.subplots(figsize=(10,4))
    sns.set_theme(style='white')

    # Gera o gráfico de colunas
    sns.barplot(data=lucro_ano, x=lucro_ano.index, y='lucro', palette=cores)

    # Personalizando o gráfico
    ax.set_title('Lucro das vendas das lojas de departamento de\n2016 a 2019',
                loc='left', fontsize=18,color=CINZA2, fontfamily='Tahoma')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.xaxis.set_tick_params(labelsize=12, labelcolor=CINZA2, labelfontfamily='Tahoma')
    sns.despine(left=True, bottom=True)

    ax.set_yticklabels([])
    for i, valor in enumerate(lucro_ano['lucro']):
        qtd = f'R${valor:,.0f}'.replace(',','.')
        offset = 1e4
        ax.text(i, valor + offset, qtd, color=CINZA2, fontsize=12, ha='center', va='center')
    
    # Define as cores do gráfico
    cores = []
    for ano in lucro_ano.index:
        if lucro_ano.loc[ano, 'lucro'] == lucro_ano.lucro.max():
            cores.append(VERDE3)
        else:
            cores.append(VERDE1)
            # Retornando o eixo

    # Adiciona texto com detalhes
    ax.text(3.5, 1e4,
            'Em $\mathbf{2019}$ o lucro\n'
           'das lojas subiu\n'
           'aproximadamente $\mathbf{14,04}$%\n'
           'em relação a 2018.',
           fontsize=14, linespacing=1.45,color=VERDE3,
           fontfamily='Tahoma')
    return ax
    
# Chama a função do gráfico de colunas
criar_grafico_colunas(cores)


## <font color=a76125> **Desafio 2**

##### <font color=#f8d0ae> **Criando o _Dataframe_**

In [ ]:
# Copia o dataframe original
top_vendas = vendas.copy()

# Obtem o top 10 em faturamento
top_vendas = top_vendas.groupby(['tipo_produto'])['vendas'].sum().sort_values(ascending=False).reset_index()
top_10 = top_vendas[:10]

# Cria um dataframe com os departamentos
depart = vendas.copy()
depart = depart[['tipo_produto','departamento']].drop_duplicates('tipo_produto')

# Faz a junção dos dois dataframes
top_10_faturamento = top_10.merge(depart, on='tipo_produto', how='left')

##### <font color=#f8d0ae> **Gerando o gráfico com uma função**

In [ ]:
def criar_grafico_barras(cores:list=[AZUL1]): 
    # Área do gráfico e tema da visualização
    fig, ax = plt.subplots(figsize= (10,4))
    sns.set_theme(style='white')

    # Seleciona as cores a serem usadas
    cores = [AMARELO1, LARANJA1, VERMELHO1]
    mapa_cores = {
        'Materiais de construção': LARANJA1,
        'Jardinagem e paisagismo': AMARELO1,
        'Automotivo': VERMELHO1
    }
        
    # Gera o gráfico de barras
    ax = sns.barplot(data=top_10_faturamento, x= 'vendas', y= 'tipo_produto', hue='departamento', palette = mapa_cores)

    # Personaliza o gráfico
    ax.set_title('Top 10 produtos com maior faturamento nas lojas de departamento\nde 2016 a 2019',
                loc='left', fontsize=18, fontfamily='Tahoma', color=CINZA2)
    ax.yaxis.set_tick_params(labelsize=12, labelfontfamily='Tahoma', labelcolor=CINZA2)
    sns.despine(left=True, bottom=True)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.legend().remove()

    # Escrevendo os valores de cada barra no gráfico
    ax.set_xticklabels([])
    for i, valor in enumerate(top_10_faturamento['vendas']):
        qtd = f'R$ {valor:,.0f}'.replace(',','.')  
        offset = 1e4
        ax.text(valor - offset, i, qtd, color= CINZA5, fontsize=9, fontweight='bold', ha='right', va='center')
    
    # Anotando um destaque no gráfico
    ax.text(0.5,0.02,
            'Os $\mathbf{pneus}$ lideram o faturamento no período,\n'
            'ultrapassando $\mathbf{1\ milhão\ de\ reais}$ em vendas,\n'
            'enquanto $\mathbf{ferramentas}$ e $\mathbf{vasos}$ aparecem como\n'
            'os demais destaques no top 3.', transform=ax.transAxes,
            fontsize=12, ha='left', va='bottom')
    return ax

criar_grafico_barras()

---